# Importing the necessary packages/modules

In [1]:
from IPython.display import display
import pandas as pd


In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None) 

# Loading and previewing the data

In [3]:
#loading the final raw
df =pd.read_csv('../data/raw/cwlagos_listings_raw.csv')
df.head()

,type,kind,price,title,location,beds,baths,agent,contact,listing_url
0,For Rent,Apartment,"₦20,000,000",4 Units of 3 Bedroom Apartments in Lekki phase 1,Lekki Phase 1,3.0,3.0,Chinenye,+234 816 911 2079,https://cwlagos.com/property/4-units-of-3-bedroom-apartments-in-lekki-phase-1-cw08223
1,For sale,Detached Duplex,"₦1,500,000,000",Luxury 5 Bedroom Fully Detached House in Lekki phase 1,Lekki Phase 1,5.0,5.0,Chinenye,+234 816 911 2079,https://cwlagos.com/property/luxury-5-bedroom-fully-detached-house-in-lekki-phase-1-cw08221
2,For Rent,Apartment,"₦45,000,000",Furnished 3 Bedroom Apartment in Ikoyi,Banana Island,3.0,3.0,Adaeze,+234 816 631 5298,https://cwlagos.com/property/furnished-3-bedroom-apartment-in-ikoyi-cw08215
3,For Rent,Apartment,"₦89,408,528",Luxury 3-Bedroom Apartment in Ikoyi,Banana Island,3.0,3.0,Adaeze,+234 816 631 5298,https://cwlagos.com/property/luxury-3-bedroom-apartment-in-ikoyi-cw08213
4,For Rent,Commercial,"₦30,000,000",4 Bedroom Apartment for Commercial Use in VI,Victoria Island,NaN,NaN,Nadi,+2349062511345,https://cwlagos.com/property/4-bedroom-apartment-for-commercial-use-in-vi-cw08220


In [4]:
print(df.shape)

(652, 10)


In [5]:
df.columns

Index(['type', 'kind', 'price', 'title', 'location', 'beds', 'baths', 'agent',
       'contact', 'listing_url'],
      dtype='str')

# Handling the duplicate entries & inconsistent naming

## handling duplicate urls(unique identifier)

In [6]:
# previewing to see the total no of duplicates
df['listing_url'].duplicated().sum()

np.int64(0)

In [7]:
""" We see that there are no duplicates in the listing_url column, which is a good unique identifier for each listing. """

' We see that there are no duplicates in the listing_url column, which is a good unique identifier for each listing. '

## handling duplicate locations

In [8]:
df['location'].value_counts().to_frame()

,count
location,
Lekki Phase 1,118
Ikoyi,106
Victoria Island,86
Ikate,56
Lekki,43
Oniru,38
Ikota,37
Banana Island,30
Osapa,25


In [9]:
# these are unique locations in lagos, but most of them are specific areas within the larger districts, so we will need to do some cleaning to group them together
district_mapping = {
    # ikoyi parent
    "Ikoyi": "Ikoyi",
    "Old Ikoyi": "Ikoyi",
    "Banana Island": "Ikoyi",
    "Parkview": "Ikoyi",
    "Osborne Foreshore": "Ikoyi",

    # Victoria Island Parent 
    "Victoria Island": "Victoria Island",
    "Eko Atlantic": "Victoria Island",
    "Oniru": "Victoria Island",

    # Lekki Parent 
    "Lekki": "Lekki",
    "Lekki Phase 1": "Lekki",
    "Ikate": "Lekki",
    "Osapa": "Lekki",
    "Ologolo": "Lekki",
    "Chevron": "Lekki",
    "Ikota": "Lekki",
    "Orchid": "Lekki",
    "Orchid, Lekki": "Lekki",
    "Pinnock Beach Estate": "Lekki",

    # Ajah Parent 
    "Ajah": "Ajah"
}

In [10]:
# now creating a new column called 'district' and mapping the locations to their respective districts
df['district'] = df['location'].map(district_mapping)
df['district'].value_counts().to_frame()

,count
district,
Lekki,335
Ikoyi,168
Victoria Island,126
Ajah,17


## previewing the "kind" column

In [11]:
#looks very consistent
df['kind'].value_counts().to_frame()

,count
kind,
Apartment,288
Detached Duplex,126
Terrace,80
Semi Detached,39
Commercial,36
Mixed-Use Land,32
Maisonette,22
Penthouse,17
Residential Land,6


## previewing the "type" column

In [12]:
df['type'].value_counts().to_frame()

,count
type,
For Rent,321
For Sale,274
Land,47
For sale,9


In [13]:
# fixing iconsistency with the sale columns
df['type'] = df['type'].replace('For sale', 'For Sale')
df['type'].value_counts().to_frame()

,count
type,
For Rent,321
For Sale,283
Land,47


## previewing the price column

In [14]:
df['price'].head()

0       ₦20,000,000
1    ₦1,500,000,000
2       ₦45,000,000
3       ₦89,408,528
4       ₦30,000,000
Name: price, dtype: str

In [16]:
# we see the price column is in string format and has the Naira symbol and commas, I'll need to clean this to convert it to a numeric format for analysis
df['price'] = df['price'].str.replace('₦', '', regex=False)
df['price'] = df['price'].str.replace(',', '', regex=False)
df['price'] = df['price'].astype(float)
df['price'].head()

ValueError: could not convert string to float: '650000000 / 1500000000'

In [17]:
#taking a look at the specific outlier
df[df['price']  == '650000000 / 1500000000']

,type,kind,price,title,location,beds,baths,agent,contact,listing_url,district
138,For Sale,Apartment,650000000 / 1500000000,"4 Bedroom Flat & 5 Bedroom Penthouse Maisonette in Oniru, Victoria Island",Victoria Island,4.0,4.0,Ifunanya,+234 706 399 0727,https://cwlagos.com/property/4-bedroom-flat-and-5-bedroom-penthouse-maisonette-in-oniru-victoria-island-cw08104,Victoria Island


In [18]:
pd.set_option('display.max_colwidth', None)

In [19]:
# checking the url to see if there are any clues about the price
display(df[df['price']  == '650000000 / 1500000000']['listing_url'])

138    https://cwlagos.com/property/4-bedroom-flat-and-5-bedroom-penthouse-maisonette-in-oniru-victoria-island-cw08104
Name: listing_url, dtype: str

In [20]:
# i see that the listing is actually two listings in one, I'll need to create the two separate listings and drop the original one
extra_listing = [{
    "title": "5 Bedroom Penthouse Maisonette",
    "location": "Oniru",
    "type": "For Sale",
    "kind": "Maisonette",
    "beds": 5.0,
    "baths": 5.0,
    "price": 1500000000,
    "district": "Victoria Island",
    "agent": "Ifunanya",
    "contact": "+234 706 399 0727",
    "listing_url": "https://cwlagos.com/property/4-bedroom-flat-and-5-bedroom-penthouse-maisonette-in-oniru-victoria-island-cw08104"
}, 
{
    "title": "4 Bedroom Flat (235sqm) new development at Oniru, Victoria Island",
    "location": "Oniru",
    "district": "Victoria Island",
    "type": "For Sale",
    "kind": "Apartment",
    "beds": 4.0,
    "baths": 4.0,
    "price": 650000000,
    "agent": "Ifunanya",
    "contact": "+234 706 399 0727",
    "listing_url": "https://cwlagos.com/property/4-bedroom-flat-and-5-bedroom-penthouse-maisonette-in-oniru-victoria-island-cw08104",
}]

extra_listing_df = pd.DataFrame(extra_listing)
df = pd.concat([df, extra_listing_df], ignore_index=True)

In [21]:
"""dropping the original listing with the combined price"""
df = df[df['price'] != '650000000 / 1500000000']
df.shape

(653, 11)

In [22]:
#finally, converting the price column to float format for analysis
df['price'] = df['price'].astype(float)
df['price'].dtype

dtype('float64')

# Handling missing values

In [23]:
df.isnull().sum().to_frame()

,0
type,1
kind,0
price,0
title,0
location,4
beds,89
baths,89
agent,0
contact,0
listing_url,0


In [24]:
# previewing the missing values
display(df[df['beds'].isna() == True].head(), df[df['beds'].isna() == True].tail())

,type,kind,price,title,location,beds,baths,agent,contact,listing_url,district
4,For Rent,Commercial,30000000.0,4 Bedroom Apartment for Commercial Use in VI,Victoria Island,NaN,NaN,Nadi,+2349062511345,https://cwlagos.com/property/4-bedroom-apartment-for-commercial-use-in-vi-cw08220,Victoria Island
13,For Rent,Commercial,300000.0,Ground Floor Office Space (230 sqm)- Lekki phase 1,Lekki Phase 1,NaN,NaN,Nadi,+2349062511345,https://cwlagos.com/property/ground-floor-office-space-230-sqm-lekki-phase-1-cw08217,Lekki
37,For Rent,Commercial,100000000.0,Commercial Building with Warehouse In VI,Victoria Island,NaN,NaN,Nadi,+2349062511345,https://cwlagos.com/property/commercial-building-with-warehouse-in-vi-cw08170,Victoria Island
42,Land,Apartment,4000000.0,Prime Land in Ikoyi,Ikoyi,NaN,NaN,Rose,+234 814 922 6187,https://cwlagos.com/property/prime-land-in-ikoyi-cwl1424,Ikoyi
47,Land,Apartment,4700000.0,Land Measuring 4876sqm in Ikoyi,Ikoyi,NaN,NaN,Rose,+234 814 922 6187,https://cwlagos.com/property/land-measuring-4876sqm-in-ikoyi-cwl1421,Ikoyi


,type,kind,price,title,location,beds,baths,agent,contact,listing_url,district
622,For Rent,Commercial,85000000.0,Detached Commercial Property- VI,Victoria Island,NaN,NaN,Nadi,+2349062511345,https://cwlagos.com/property/detached-commercial-property-vi-cw07910,Victoria Island
636,For Rent,Commercial,618982.0,Grade A Office Space - Victoria Island,Victoria Island,NaN,NaN,Nadi,+2349062511345,https://cwlagos.com/property/grade-a-office-space-victoria-island-cw07678,Victoria Island
647,For Rent,Commercial,130000000.0,Waterfront Commercial Development – Lekki Phase 1,Lekki Phase 1,NaN,NaN,Nadi,+2349062511345,https://cwlagos.com/property/waterfront-commercial-development-lekki-phase-1-cw07623,Lekki
650,For Rent,Commercial,825309.0,GRADE A COMMERCIAL BUILDING– IKOYI,Ikoyi,NaN,NaN,Nadi,+2349062511345,https://cwlagos.com/property/grade-a-commercial-building-ikoyi-cw07679,Ikoyi
651,For Rent,Commercial,25000000.0,Upper-Floor Commercial Space in Lekki Phase 1,Lekki Phase 1,NaN,NaN,Nadi,+2349062511345,https://cwlagos.com/property/upper-floor-commercial-space-in-lekki-phase-1-cw07951,Lekki


In [25]:
# we see that the those listings with no beds and baths are mostly plots of land or commercial properties, so we will replace the missing values with 0
df['beds'] = df['beds'].fillna(0)
df['baths'] = df['baths'].fillna(0)
# checking the result
df[df['beds'].isna() == True].head()



,type,kind,price,title,location,beds,baths,agent,contact,listing_url,district


In [26]:
#previewing the missing location values
display(df[df['location'].isna() == True])

,type,kind,price,title,location,beds,baths,agent,contact,listing_url,district
373,For Sale,Apartment,190000000.0,2 Bedroom Apartment in Lekki,NaN,2.0,2.0,Peter,+234 906 251 1344,https://cwlagos.com/property/2-bedroom-apartment-in-lekki-cw07975,NaN
446,Land,Residential Land,15000000.0,600SQM Land in an Estate- Ibeju Lekki,NaN,0.0,0.0,Emmanuel,+234 908 807 2106,https://cwlagos.com/property/600sqm-land-in-an-estate-ibeju-lekki-cwl1391,NaN
555,NaN,Mixed-Use Land,3438790.0,2245sqm Development Land- VI,NaN,0.0,0.0,Emmanuel,+234 908 807 2106,https://cwlagos.com/property/2245sqm-development-land-vi-cwl1392,NaN
616,For Rent,Maisonette,43400000.0,3 Bedroom Maisonette in Lekki,NaN,3.0,3.0,Peter,+234 906 251 1344,https://cwlagos.com/property/3-bedroom-maisonette-in-lekki-cw07976,NaN


In [27]:
# replacing the values for the missing values after previewing the listing urls
df.loc[[373, 446, 616], 'location'] = 'Lekki'
df.loc[[555], 'location'] = 'Victoria Island'

df.loc[[373, 446, 616], 'district'] = 'Lekki'
df.loc[[555], 'district'] = 'Victoria Island'

df.loc[[555], 'type'] = 'Land'


In [28]:
#previewing the missing district values
display(df[df['district'].isna() == True])

,type,kind,price,title,location,beds,baths,agent,contact,listing_url,district
34,For Sale,Detached Duplex,400000000.0,5-Bedroom Fully Detached Duplex,Lekki county,5.0,5.0,Peter,+234 906 251 1344,https://cwlagos.com/property/5-bedroom-fully-detached-duplex-cw08207,NaN
257,For Sale,Detached Duplex,400000000.0,5 Bedroom Fully Detached Duplex in Lekki,Idado,5.0,5.0,Peter,+234 906 251 1344,https://cwlagos.com/property/5-bedroom-fully-detached-duplex-in-lekki-cw07608,NaN


In [29]:
df.loc[[34,257], 'district'] = 'Lekki'

In [30]:
df.isnull().sum().to_frame()

,0
type,0
kind,0
price,0
title,0
location,0
beds,0
baths,0
agent,0
contact,0
listing_url,0


# Final review of the cleaned data

In [31]:
df.describe(include='all')

,type,kind,price,title,location,beds,baths,agent,contact,listing_url,district
count,653,653,6.530000e+02,653,653,653.000000,653.000000,653,653,653,653
unique,3,11,NaN,505,21,NaN,NaN,17,17,652,4
top,For Rent,Apartment,NaN,2 Bedroom Apartment in Lekki,Lekki Phase 1,NaN,NaN,Jennifer,+234 704 808 9361,https://cwlagos.com/property/4-bedroom-flat-and-5-bedroom-penthouse-maisonette-in-oniru-victoria-island-cw08104,Lekki
freq,321,288,NaN,11,118,NaN,NaN,89,89,2,340
mean,NaN,NaN,4.379179e+08,NaN,NaN,3.093415,3.093415,NaN,NaN,NaN,NaN
std,NaN,NaN,1.110910e+09,NaN,NaN,2.461763,2.461763,NaN,NaN,NaN,NaN
min,NaN,NaN,1.500000e+05,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,2.500000e+07,NaN,NaN,2.000000,2.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,1.200000e+08,NaN,NaN,3.000000,3.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,4.500000e+08,NaN,NaN,4.000000,4.000000,NaN,NaN,NaN,NaN


In [32]:
df = df.drop(columns=["contact"])

In [33]:
df.head()

,type,kind,price,title,location,beds,baths,agent,listing_url,district
0,For Rent,Apartment,2.000000e+07,4 Units of 3 Bedroom Apartments in Lekki phase 1,Lekki Phase 1,3.0,3.0,Chinenye,https://cwlagos.com/property/4-units-of-3-bedroom-apartments-in-lekki-phase-1-cw08223,Lekki
1,For Sale,Detached Duplex,1.500000e+09,Luxury 5 Bedroom Fully Detached House in Lekki phase 1,Lekki Phase 1,5.0,5.0,Chinenye,https://cwlagos.com/property/luxury-5-bedroom-fully-detached-house-in-lekki-phase-1-cw08221,Lekki
2,For Rent,Apartment,4.500000e+07,Furnished 3 Bedroom Apartment in Ikoyi,Banana Island,3.0,3.0,Adaeze,https://cwlagos.com/property/furnished-3-bedroom-apartment-in-ikoyi-cw08215,Ikoyi
3,For Rent,Apartment,8.940853e+07,Luxury 3-Bedroom Apartment in Ikoyi,Banana Island,3.0,3.0,Adaeze,https://cwlagos.com/property/luxury-3-bedroom-apartment-in-ikoyi-cw08213,Ikoyi
4,For Rent,Commercial,3.000000e+07,4 Bedroom Apartment for Commercial Use in VI,Victoria Island,0.0,0.0,Nadi,https://cwlagos.com/property/4-bedroom-apartment-for-commercial-use-in-vi-cw08220,Victoria Island


In [35]:
df.to_csv('../data/processed/lagos_real_estate_market_data_cleaned.csv', index=False)